# 02 · Purchase Prediction

**Business question:** *Which customers will buy again in the next 30 days?*

Knowing this lets marketing target reminders and offers at the customers who need them most.

**Setting up the problem without data leakage**

```
        history (used to build features)            future (used for the label)
  ... ---------------------------------|----------------------------------|
                                 2011-11-09           2011-11-10  →  2011-12-09
```

* **Features** (RFM) are computed only from purchases up to **2011-11-09** (`vw_PurchasePredictionFeatures`).
* **Label** `PurchaseNext30Days = 1` if the customer bought between **2011-11-10 and 2011-12-09** (`vw_PurchasePredictionTarget`).

Because the model never sees information from the period it is asked to predict, the evaluation is honest.

**Models compared:** Logistic Regression, Random Forest, XGBoost.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Find the project root (the folder that contains "src"), whatever the working directory is
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "db.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open this notebook from inside the retail-intelligence folder (it needs the src/ folder next to it).")
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.db import read_sql       # runs a SELECT on SQL Server and returns a DataFrame

plt.rcParams["figure.dpi"] = 100

import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

RANDOM_STATE = 42

## 2. Build the modelling table

In [ ]:
df_features = read_sql("SELECT * FROM dbo.vw_PurchasePredictionFeatures")
df_target = read_sql("SELECT * FROM dbo.vw_PurchasePredictionTarget")

# The target view only lists customers who DID buy; everyone else gets 0.
df = df_features.merge(df_target, on="CustomerID", how="left")
df["PurchaseNext30Days"] = df["PurchaseNext30Days"].fillna(0).astype(int)

print("Customers:", df.shape[0])
df.head()

### Class balance

Only about one customer in four buys in the next 30 days, so the classes are imbalanced. Accuracy alone would be misleading, so precision, recall, F1 and ROC-AUC are used as well.

In [ ]:
balance = df["PurchaseNext30Days"].value_counts().to_frame("customers")
balance["share_%"] = balance["customers"] / balance["customers"].sum() * 100
balance

## 3. Train / test split

* 80 % train, 20 % test.
* `stratify` keeps the same buyer / non-buyer ratio in both parts.
* Logistic Regression needs scaled inputs (log + standardise); tree models do not.

In [ ]:
FEATURES = ["Recency", "Frequency", "Monetary", "TotalQuantity", "UniqueProducts", "AOV"]

X = df[FEATURES]
y = df["PurchaseNext30Days"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scaling is fitted on the TRAIN set only and then applied to the test set.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(np.log1p(X_train))
X_test_scaled = scaler.transform(np.log1p(X_test))

print("Train:", X_train.shape, "| Test:", X_test.shape)

## 4. Train the three models

In [ ]:
logistic = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
logistic.fit(X_train_scaled, y_train)

forest = RandomForestClassifier(
    n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
)
forest.fit(X_train, y_train)

booster = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, eval_metric="logloss",
)
booster.fit(X_train, y_train)

# model -> (fitted model, matching test matrix)
models = {
    "Logistic Regression": (logistic, X_test_scaled),
    "Random Forest": (forest, X_test),
    "XGBoost": (booster, X_test),
}

## 5. Compare the models  (default decision threshold = 0.5)

In [ ]:
probabilities = {}
rows = []

for name, (model, X_eval) in models.items():
    y_pred = model.predict(X_eval)
    y_prob = model.predict_proba(X_eval)[:, 1]
    probabilities[name] = y_prob

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })

comparison = pd.DataFrame(rows).set_index("Model")
comparison.round(4)

In [ ]:
plt.figure(figsize=(7, 6))

for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc_score(y_test, y_prob):.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random guess")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curves")
plt.legend()
plt.tight_layout()
plt.show()

**Reading the results**

* The three models are close; ROC-AUC (how well they *rank* buyers above non-buyers) is highest for Logistic Regression, with XGBoost nearly tied.
* At the default 0.5 threshold recall is low: the models miss most of the real buyers. That is a threshold problem, not a ranking problem, which is addressed in section 7.
* With so few features and a small number of customers, a simple, interpretable model is a sensible final choice.

## 6. What drives the predictions?

In [ ]:
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Random Forest": forest.feature_importances_,
    "XGBoost": booster.feature_importances_,
}).set_index("Feature")

display(importance.sort_values("XGBoost", ascending=False).round(3))

importance.sort_values("XGBoost").plot(kind="barh", figsize=(9, 4.5))
plt.title("Feature importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

In the original run, Recency is among the top two features in both tree models. XGBoost relies most on Frequency, while Random Forest spreads the rest of its importance fairly evenly across the other features. Importance shows *how much* a feature is used, not in which direction it pushes the prediction.

## 7. Choose the decision threshold

The business goal is usually to **catch as many likely buyers as possible**, so a lower threshold than 0.5 is worth testing. The table shows the trade-off for Logistic Regression: lowering the threshold raises recall and lowers precision.

In [ ]:
def threshold_table(y_true, y_prob, thresholds=np.arange(0.20, 0.81, 0.05)):
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        rows.append({
            "Threshold": round(t, 2),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
        })
    return pd.DataFrame(rows)


logistic_thresholds = threshold_table(y_test, probabilities["Logistic Regression"])
display(logistic_thresholds.round(4))

best = logistic_thresholds.loc[logistic_thresholds["F1"].idxmax()]
print(f"Best F1 at threshold {best['Threshold']:.2f}: "
      f"precision = {best['Precision']:.3f}, recall = {best['Recall']:.3f}, F1 = {best['F1']:.3f}")

**Final choice: Logistic Regression with threshold 0.25.**

At 0.25 the model finds about 7 in 10 real buyers (recall) while roughly half of the customers it flags do buy (precision). That is a good fit for a low-cost reminder campaign, where contacting a few extra customers costs little but missing a buyer costs revenue.

> **Limitation:** the threshold was selected on the same test set that is used to report the result, so the final numbers are slightly optimistic. The proper fix is to choose the threshold on a separate validation set or with cross-validation (see the project README).

## 8. Final model evaluation

In [ ]:
FINAL_THRESHOLD = 0.25

final_prob = probabilities["Logistic Regression"]
final_pred = (final_prob >= FINAL_THRESHOLD).astype(int)

print(f"Logistic Regression, threshold = {FINAL_THRESHOLD}\n")
print(classification_report(y_test, final_pred, target_names=["No purchase", "Purchase"]))

ConfusionMatrixDisplay.from_predictions(
    y_test, final_pred, display_labels=["No purchase", "Purchase"], cmap="Blues"
)
plt.title(f"Confusion matrix (threshold = {FINAL_THRESHOLD})")
plt.tight_layout()
plt.show()

### Using the model: customers to contact first

In [ ]:
scored = X_test.copy()
scored["ActualPurchase"] = y_test.values
scored["PurchaseProbability"] = final_prob
scored["PredictedPurchase"] = final_pred

# Highest probability first -> the customers a campaign should prioritise
scored.sort_values("PurchaseProbability", ascending=False).head(10)

## 9. Export for the web app

The final model is a Logistic Regression, so it can be saved as plain numbers (scaler mean/scale, coefficients, intercept, threshold) in a JSON file. The app recomputes the probability with a few lines of NumPy: no pickle file, no scikit-learn version problems, and the model stays easy to inspect.

In [ ]:
import json

ARTIFACTS = root / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

# 1) The model as plain numbers
model_card = {
    "model": "Logistic Regression",
    "features": FEATURES,
    "preprocessing": "log1p, then standardise with the training mean and scale",
    "scaler_mean": scaler.mean_.tolist(),
    "scaler_scale": scaler.scale_.tolist(),
    "coefficients": logistic.coef_[0].tolist(),
    "intercept": float(logistic.intercept_[0]),
    "threshold": FINAL_THRESHOLD,
    "cutoff_date": "2011-11-09",
    "horizon_days": 30,
    "test_metrics": {
        "precision": float(precision_score(y_test, final_pred)),
        "recall": float(recall_score(y_test, final_pred)),
        "f1": float(f1_score(y_test, final_pred)),
        "roc_auc": float(roc_auc_score(y_test, final_prob)),
    },
}
with open(ARTIFACTS / "purchase_model.json", "w", encoding="utf-8") as f:
    json.dump(model_card, f, indent=2)

# 2) Every customer scored as of the cut-off date
all_prob = logistic.predict_proba(scaler.transform(np.log1p(df[FEATURES])))[:, 1]
scores = df[["CustomerID"] + FEATURES].copy()
scores["PurchaseProbability"] = all_prob
scores["PredictedPurchase"] = (all_prob >= FINAL_THRESHOLD).astype(int)
scores.to_csv(ARTIFACTS / "customer_purchase_scores.csv", index=False)

print("Saved:", sorted(f.name for f in ARTIFACTS.glob("*")))